# Exercícios — SQL Básico com PySpark

Este notebook traz **13 exercícios** de fixação: o primeiro é sobre criação e alteração de tabelas do zero, e os outros 12 usam a base de dados de livros (`livros_vendas.csv`), sendo 4 exercícios básicos, 4 intermediários e 4 que exigem um pouco mais de lógica (subqueries, window functions e CTEs).

Cada exercício tem: o **enunciado** (o que você precisa fazer), seguido de uma célula de código com a **resolução comentada**. A sugestão é tentar resolver sozinho antes de olhar a célula de resposta!

Rode a célula abaixo primeiro para iniciar a sessão Spark e carregar a base de livros.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, BinaryType, ArrayType, MapType, DecimalType
)
import datetime
import decimal
from pyspark.sql.functions import col
from pyspark.sql.functions import lit

spark = SparkSession.builder.getOrCreate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
caminho = "/content/drive/MyDrive/{caminho_ate_o_arquivo}/livros_vendas.csv"
df = spark.read.csv(caminho, header=True, inferSchema=True)
df.printSchema()

## Exercício 1 — Criando e alterando seu próprio banco de dados

Antes de trabalharmos com a base de livros, pratique os comandos de criação e alteração de tabela usando um tema **de sua escolha** (times de futebol, jogos, animais, filmes, o que preferir).

**a)** Crie uma tabela com no mínimo 4 colunas e 3 linhas;

**b)** Renomeie uma das colunas da tabela;

**c)** Altere o tipo de dado de uma das colunas;

**d)** Adicione uma coluna à tabela;

**e)** Delete a tabela.

Abaixo está uma resolução de exemplo, usando uma tabela de animais de estimação — a ideia é que a sua tabela, sobre o tema que você escolher, siga essa mesma estrutura de comandos.

In [ ]:
# a) Criar uma tabela e inserir valores

spark.sql("""
    CREATE TABLE animais (
        id INT,
        nome VARCHAR(30),
        especie VARCHAR(30),
        idade int
        )
    """)

spark.sql("""
    INSERT INTO animais VALUES
        (1, 'Camomila', 'gato', '1'),
        (2, 'William King', 'hamster', '3'),
        (3, 'Teco', 'calopsita', '7')
        """)

In [ ]:
# b) Renomeie uma das colunas

df = df.withColumnRenamed("coluna_antiga", "coluna_nova")

In [ ]:
# c) Altere o tipo de dado de uma das colunas

df = df.withColumn("coluna", col("coluna").cast("tipo"))

In [ ]:
# d) Adicione uma coluna à tabela

df = df.withColumn("nova_coluna")

In [ ]:
# e) Delete a tabela

spark.sql("DROP TABLE tabela")

## Exercício 2 — SELECT com filtro simples

Selecione as colunas `nome`, `autor` e `valor` de todos os livros do gênero `'Fantasia'`.

In [ ]:
spark.sql("""
    SELECT nome, autor, valor
    FROM livros
    WHERE genero = 'Fantasia'
""").show()

## Exercício 3 — Ordenando e limitando resultados

Liste os 10 livros com o maior número de `paginas`, do maior para o menor.

In [ ]:
spark.sql("""
    SELECT nome, autor, paginas
    FROM livros
    ORDER BY paginas DESC
    LIMIT 10
""").show()

## Exercício 4 — Contagem por grupo

Quantos livros distintos existem em cada `continente`?

In [ ]:
spark.sql("""
    SELECT continente, COUNT(*) AS qtd_livros
    FROM livros
    GROUP BY continente
    ORDER BY qtd_livros DESC
""").show()

## Exercício 5 — Média por grupo

Qual o `valor` médio dos livros de cada `genero`? Ordene do gênero mais caro (em média) para o mais barato, e arredonde a média para 2 casas decimais.

In [ ]:
spark.sql("""
    SELECT genero, ROUND(AVG(valor), 2) AS valor_medio
    FROM livros
    GROUP BY genero
    ORDER BY valor_medio DESC
""").show()

## Exercício 6 — Combinando filtros

Encontre os livros com **mais de 500 páginas** e `valor` **abaixo de R$ 40**. Ordene do maior para o menor número de páginas.

> Você pode combinar duas condições no `WHERE` usando `AND` — igual fazemos em português: "página > 500 **e** valor < 40".

In [ ]:
spark.sql("""
    SELECT nome, autor, paginas, valor
    FROM livros
    WHERE paginas > 500 AND valor < 40
    ORDER BY paginas DESC
""").show()

## Exercício 7 — Arredondando um cálculo entre colunas

Para cada livro, calcule a receita gerada (`valor * unidades_vendidas`). Mostre os 5 livros que mais geraram receita, com o resultado arredondado para 2 casas decimais.

In [ ]:
spark.sql("""
    SELECT nome, autor, ROUND(valor * unidades_vendidas, 2) AS receita
    FROM livros
    ORDER BY receita DESC
    LIMIT 5
""").show()

## Exercício 8 — Contagem por grupo, em ordem

Liste a quantidade de livros de cada `genero`, ordenado do gênero com mais livros para o com menos.

Depois de rodar a consulta, **observe o resultado** e responda: quais gêneros têm mais de 10 livros na base?

In [ ]:
spark.sql("""
    SELECT genero, COUNT(*) AS qtd_livros
    FROM livros
    GROUP BY genero
    ORDER BY qtd_livros DESC
""").show(30)

## Exercício 9 — Comparando faixas de preço

Sem criar nenhuma coluna nova, descubra: quantos livros custam **menos de R$ 40**? Quantos custam **entre R$ 40 e R$ 70**? Quantos custam **mais de R$ 70**?

> Dica: são três consultas separadas, cada uma usando `WHERE` + `COUNT`.

In [ ]:
spark.sql("SELECT COUNT(*) AS livros_ate_40 FROM livros WHERE valor < 40").show()

spark.sql("""
    SELECT COUNT(*) AS livros_entre_40_e_70
    FROM livros
    WHERE valor >= 40 AND valor <= 70
""").show()

spark.sql("SELECT COUNT(*) AS livros_acima_70 FROM livros WHERE valor > 70").show()

## Exercício 10 — Encontrando o livro mais vendido de cada continente (em duas etapas)

Descubra qual foi o livro **mais vendido** (`unidades_vendidas`) de cada `continente`.

Como só temos `GROUP BY` com funções de agregação (`MAX`, `MIN`, `AVG`, `COUNT`) — e essas funções não retornam o `nome` do livro, só o número — vamos resolver em **duas etapas**:

1. Primeiro, descubra o **maior número de `unidades_vendidas`** de cada continente (`GROUP BY` + `MAX`).
2. Depois, para cada continente, use o número que você encontrou no passo 1 dentro de um `WHERE` (`continente = '...' AND unidades_vendidas = ...`) para descobrir qual livro é.

In [ ]:
# Etapa 1: maior número de unidades vendidas por continente
spark.sql("""
    SELECT continente, MAX(unidades_vendidas) AS max_vendas
    FROM livros
    GROUP BY continente
    ORDER BY max_vendas DESC
""").show()

In [ ]:
# Etapa 2: usamos os valores encontrados acima para achar o livro correspondente,
# um continente de cada vez.

spark.sql("""
    SELECT nome, autor, continente, unidades_vendidas
    FROM livros
    WHERE continente = 'América do Sul' AND unidades_vendidas = 194429
""").show()

spark.sql("""
    SELECT nome, autor, continente, unidades_vendidas
    FROM livros
    WHERE continente = 'América do Norte' AND unidades_vendidas = 175746
""").show()

spark.sql("""
    SELECT nome, autor, continente, unidades_vendidas
    FROM livros
    WHERE continente = 'Europa' AND unidades_vendidas = 175490
""").show()

spark.sql("""
    SELECT nome, autor, continente, unidades_vendidas
    FROM livros
    WHERE continente = 'África' AND unidades_vendidas = 101012
""").show()

spark.sql("""
    SELECT nome, autor, continente, unidades_vendidas
    FROM livros
    WHERE continente = 'Ásia' AND unidades_vendidas = 60311
""").show()

spark.sql("""
    SELECT nome, autor, continente, unidades_vendidas
    FROM livros
    WHERE continente = 'Oceania' AND unidades_vendidas = 31776
""").show()

## Exercício 11 — Amplitude de preço por autor

Para cada `autor`, calcule a diferença entre o `valor` do livro mais caro e o do mais barato (`MAX(valor) - MIN(valor)`).

Depois de rodar, observe o resultado: quais autores tiveram diferença igual a `0`? O que isso te diz sobre esses autores — eles têm só um livro na base, ou têm vários livros com o mesmo valor?

In [ ]:
spark.sql("""
    SELECT autor, ROUND(MAX(valor) - MIN(valor), 2) AS amplitude_preco
    FROM livros
    GROUP BY autor
    ORDER BY amplitude_preco DESC
""").show(30)

# Resposta: autores com amplitude = 0 são, na maioria, autores com apenas 1 livro na
# base (não há outro valor para comparar). Vale conferir com um COUNT(*) se quiser
# confirmar quantos livros cada um desses autores tem.

## Exercício 12 — Receita por página, por gênero

Descubra qual `genero` tem, em média, a maior **receita por página** — ou seja, a média de `(valor * unidades_vendidas) / paginas` entre os livros daquele gênero.

In [ ]:
spark.sql("""
    SELECT
        genero,
        ROUND(AVG((valor * unidades_vendidas) / paginas), 2) AS receita_media_por_pagina
    FROM livros
    GROUP BY genero
    ORDER BY receita_media_por_pagina DESC
""").show()